# AFOLU Flux Estimation from Gridded Extension

Estimate CO2 fluxes from Agriculture, Forestry and Other Land Use (AFOLU) implied by our gridded LUH2 extension for **REMIND-MAgPIE 3.5-4.11 / SSP1 - Very Low Emissions**.

**Goal**: Verify that the gridded extension produces land-use change fluxes consistent with the target of **net-zero AFOLU CO2 emissions by 2149** from the IAM CSV.

**Approach**: We don't have spatially-resolved carbon density fields, so we use a simplified bookkeeping method:
1. Compute per-cell year-on-year land-use transitions (area gained/lost per land type)
2. Apply representative carbon densities per land type (from literature) to convert area changes to CO2 fluxes
3. Weight by grid-cell area and sum globally
4. Compare the resulting timeseries against the IAM AFOLU trajectory

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from src import config as cfg
from src.data import (
    load_csv, filter_scenario, get_variable,
    load_states, state_2100, rates_2100,
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)

## 1. Load IAM Target Trajectory

Extract the AFOLU CO2 emissions from the CSV. This is our target: the extension should produce fluxes whose *shape* matches this curve, reaching zero by 2149.

In [ ]:
df = load_csv()
sc = filter_scenario(df)
afolu_target = get_variable(sc, 'Emissions|CO2|AFOLU')

fig, ax = plt.subplots(figsize=(12, 4))
afolu_target.loc[2050:2250].plot(ax=ax, marker='.', ms=3)
ax.axhline(0, color='k', ls='--', lw=0.5)
ax.axvline(cfg.YR_AFOLU_ZERO, color='r', ls=':', label=f'AFOLU=0 @ {cfg.YR_AFOLU_ZERO}')
ax.axvline(cfg.YR_END_INPUT, color='gray', ls='--', label='End of input (2100)')
ax.set_ylabel('Mt CO2/yr'); ax.set_title('IAM AFOLU CO2 target trajectory')
ax.legend(); plt.tight_layout(); plt.show()

print(f"AFOLU at 2100: {afolu_target.loc[2100]:.1f} Mt CO2/yr")
print(f"AFOLU at 2149: {afolu_target.loc[2149]:.1f} Mt CO2/yr")
print(f"AFOLU at 2150: {afolu_target.loc[2150]:.1f} Mt CO2/yr")

## 2. Load Extended State Fields

Open the gridded extension output from notebook 02 and the original LUH2 input, along with grid-cell areas.

The states extension file contains only the **ramp period** (2101-2150) since all land-use change ceases at 2150 (states frozen). Post-ramp AFOLU flux is therefore identically zero.

In [ ]:
# Extension output (ramp period only: 2101-2150)
STATES_STEM = cfg.STATES_FILE.stem
ext_fname = f'{STATES_STEM}_extension_transient_2101-2150.nc'
ext_path = cfg.OUTPUT_DIR / ext_fname
ds_ext = xr.open_dataset(ext_path)

# Original input (for the pre-2100 period and join continuity)
ds_input = load_states()

lat = ds_ext.lat.values
lon = ds_ext.lon.values
ramp_years = ds_ext.time.values.astype(int)  # 2101-2150
# Full extension range for plotting (post-ramp flux is zero)
full_ext_years = np.arange(cfg.YR_END_INPUT + 1, cfg.YR_END_OUTPUT + 1)

# Grid-cell area (km2) from latitude — approximate spherical Earth
R_EARTH = 6371.0  # km
dlat = np.abs(np.diff(lat[:2]))[0]  # degrees
dlon = np.abs(np.diff(lon[:2]))[0]
lat_rad = np.deg2rad(lat)
cell_area = (R_EARTH**2) * np.deg2rad(dlat) * np.deg2rad(dlon) * np.cos(lat_rad)
# Broadcast to 2-D (lat, lon)
cell_area_2d = np.broadcast_to(cell_area[:, np.newaxis], (len(lat), len(lon)))

print(f"Ramp period: {ramp_years[0]}-{ramp_years[-1]}, {len(ramp_years)} timesteps")
print(f"Full extension: {full_ext_years[0]}-{full_ext_years[-1]}, {len(full_ext_years)} years")
print(f"Grid: {len(lat)} x {len(lon)}")
print(f"Cell area range: {cell_area.min():.1f} - {cell_area.max():.1f} km2")

## 3. Carbon Density Assumptions

Define representative above-ground carbon densities per land type (tC/km2). These are rough global averages from IPCC AR6 / Houghton & Nassikas (2017) / similar literature. The absolute magnitude of the estimated flux depends on these values, but the *shape and timing* of the trajectory does not.

| Category | Representative C density (tC/km2) | Notes |
|----------|:-:|-------|
| Primary forest | 15,000 | Tropical + temperate/boreal mix |
| Secondary forest | 8,000 | Younger, lower biomass than primary |
| Primary non-forest | 1,500 | Grassland/shrubland with some woody biomass |
| Secondary non-forest | 1,000 | Recovering non-forest |
| Cropland (all types) | 500 | Low standing biomass |
| Pasture | 800 | Managed grassland |
| Rangeland | 600 | Extensive grazing land |
| Urban | 200 | Minimal vegetation |
| Plantations | 6,000 | Managed timber, moderate density |

When land is converted *from* a high-C type *to* a low-C type the difference is emitted. Conversely, land reverting to forest sequesters carbon. We compute:

$$\text{Flux}_{cell} = \sum_v \Delta f_v \times C_v \times A_{cell}$$

where $\Delta f_v$ is the year-on-year change in fraction $v$, $C_v$ is the carbon density, and $A_{cell}$ is the cell area. The sign convention is: **positive = emission, negative = sequestration** (matching the IAM convention).

In [ ]:
# Carbon densities: tC per km2 of land area (above-ground + soil, rough global mean)
# Sign convention for the density: higher density on forested land means
# *clearing* forest emits carbon and *regrowing* forest sequesters it.
# The flux formula: sum over vars of (-delta_frac * C_density * cell_area)
# Negative delta_frac (land type shrinking) on a high-C type => positive flux (emission)
# We negate so that loss of forest = positive emission.

CARBON_DENSITY = {
    # tC / km2
    'primf': 15_000,
    'secdf':  8_000,
    'primn':  1_500,
    'secdn':  1_000,
    'c3ann':    500,
    'c3nfx':    500,
    'c3per':    500,
    'c4ann':    500,
    'c4per':    500,
    'pastr':    800,
    'range':    600,
    'urban':    200,
    'pltns':  6_000,
}

# Conversion: tC -> Mt CO2
# 1 tC = 44/12 tCO2 = 3.667 tCO2
# 1 Mt = 1e6 t
TC_TO_MTCO2 = 3.667 / 1e6

print("Carbon densities (tC/km2):")
for v, c in CARBON_DENSITY.items():
    print(f"  {v:8s}: {c:>8,}")

## 4. Compute AFOLU Fluxes from the Input Period (Pre-2100)

Calculate the implied AFOLU CO2 flux from the original LUH2 input for 2024-2100 using the same bookkeeping method. This gives us a baseline to compare against the IAM trajectory and calibrate our estimate before looking at the extension.

In [ ]:
%%time
input_years = ds_input.time.values.astype(int)
n_input = len(input_years)

# Compute year-on-year flux for the input period
input_flux = np.zeros(n_input - 1)
input_flux_years = input_years[1:]  # 2025-2100

for ti in range(1, n_input):
    annual_flux = 0.0
    for v in cfg.STATE_VARS:
        delta = ds_input[v].isel(time=ti).values - ds_input[v].isel(time=ti-1).values
        # Loss of high-C land = emission (positive), so negate delta for carbon accounting
        # When a high-C type shrinks (delta < 0), carbon is released (positive flux)
        flux_v = np.nansum(-delta * CARBON_DENSITY[v] * cell_area_2d) * TC_TO_MTCO2
        annual_flux += flux_v
    input_flux[ti - 1] = annual_flux

print(f"Input flux computed for {len(input_flux_years)} transitions ({input_flux_years[0]}-{input_flux_years[-1]})")
print(f"Flux range: {input_flux.min():.1f} to {input_flux.max():.1f} Mt CO2/yr")

## 5. Compute AFOLU Fluxes from the Extension Period (2101-2500)

Same bookkeeping method applied to the extended state fields. The first transition (2100->2101) bridges the input and extension datasets. After the ramp ends at 2150, all rates are zero so fluxes are identically zero for 2151-2500.

In [ ]:
%%time
n_ramp = len(ramp_years)
ramp_flux = np.zeros(n_ramp)

# First year: transition from input 2100 to extension 2101
vals_2100 = {v: ds_input[v].isel(time=-1).values for v in cfg.STATE_VARS}

for ti in range(n_ramp):
    annual_flux = 0.0
    for v in cfg.STATE_VARS:
        if ti == 0:
            prev = vals_2100[v]
        else:
            prev = ds_ext[v].isel(time=ti-1).values
        curr = ds_ext[v].isel(time=ti).values
        delta = curr - prev
        flux_v = np.nansum(-delta * CARBON_DENSITY[v] * cell_area_2d) * TC_TO_MTCO2
        annual_flux += flux_v
    ramp_flux[ti] = annual_flux

# Full extension flux: ramp period + zeros for post-ramp (no land-use change)
ext_flux = np.zeros(len(full_ext_years))
ext_flux[:n_ramp] = ramp_flux

print(f"Ramp flux computed for {n_ramp} years ({ramp_years[0]}-{ramp_years[-1]})")
print(f"Flux range: {ramp_flux.min():.1f} to {ramp_flux.max():.1f} Mt CO2/yr")
print(f"Flux at 2101: {ramp_flux[0]:.1f} Mt CO2/yr")
idx_2149 = np.searchsorted(ramp_years, 2149)
print(f"Flux at 2149: {ramp_flux[idx_2149]:.1f} Mt CO2/yr")
print(f"Flux at 2150: {ramp_flux[-1]:.1f} Mt CO2/yr")
print(f"Post-ramp flux (2151-2500): 0.0 by construction")

## 6. Compare Estimated Fluxes vs IAM Target

The key diagnostic: overlay our bookkeeping estimate on the IAM AFOLU trajectory.

**What to expect:**
- The *shape* should match - declining toward zero by ~2149
- The *magnitude* will differ because our carbon densities are rough global averages, not spatially resolved
- We normalise both curves (relative to their 2100 values) to focus on the temporal shape

In [ ]:
# Combine input + extension flux
all_flux_years = np.concatenate([input_flux_years, full_ext_years])
all_flux = np.concatenate([input_flux, ext_flux])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Left panel: absolute values ---
ax = axes[0]
ax.plot(all_flux_years, all_flux, 'b-', lw=1.5, label='Bookkeeping estimate')
afolu_plot = afolu_target.loc[2025:2250]
ax.plot(afolu_plot.index, afolu_plot.values, 'r--', lw=1.5, label='IAM target (AFOLU CO2)')
ax.axhline(0, color='k', ls='-', lw=0.5)
ax.axvline(cfg.YR_END_INPUT, color='gray', ls='--', lw=0.8, label='End of input')
ax.axvline(cfg.YR_AFOLU_ZERO, color='r', ls=':', lw=0.8, label=f'AFOLU=0 @ {cfg.YR_AFOLU_ZERO}')
ax.set_xlabel('Year'); ax.set_ylabel('Mt CO2/yr')
ax.set_title('AFOLU flux: estimate vs IAM target')
ax.legend(fontsize=9)
ax.set_xlim(2050, 2250)

# --- Right panel: normalised to 2100 ---
ax = axes[1]
# Get estimate flux near 2100 (average of last few input years to smooth)
est_2100 = np.mean(input_flux[-5:])
afolu_2100 = afolu_target.loc[2100]

# Extension period only (post-2100)
ax.plot(full_ext_years[:150], ext_flux[:150] / est_2100, 'b-', lw=1.5, label='Estimate (normalised)')

afolu_ext = afolu_target.loc[2101:2250]
ax.plot(afolu_ext.index, afolu_ext.values / afolu_2100, 'r--', lw=1.5,
        label='IAM target (normalised)')

ax.axhline(0, color='k', ls='-', lw=0.5)
ax.axvline(cfg.YR_AFOLU_ZERO, color='r', ls=':', lw=0.8)
ax.set_xlabel('Year'); ax.set_ylabel('Fraction of 2100 value')
ax.set_title('Normalised shape comparison (post-2100)')
ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

print(f"\nEstimated flux at 2100 (avg last 5 input yrs): {est_2100:.1f} Mt CO2/yr")
print(f"IAM AFOLU at 2100: {afolu_2100:.1f} Mt CO2/yr")
print(f"Scale ratio (IAM / estimate): {afolu_2100 / est_2100:.2f}")

## 7. Flux Decomposition by Land Type

Break down the total AFOLU flux by land-use category to understand which transitions contribute most to emissions and sequestration during the extension period.

In [ ]:
%%time
# Per-variable flux decomposition for the ramp period only (post-ramp is zero)
flux_by_var = {v: np.zeros(len(full_ext_years)) for v in cfg.STATE_VARS}

for ti in range(n_ramp):
    for v in cfg.STATE_VARS:
        if ti == 0:
            prev = vals_2100[v]
        else:
            prev = ds_ext[v].isel(time=ti-1).values
        curr = ds_ext[v].isel(time=ti).values
        delta = curr - prev
        flux_by_var[v][ti] = np.nansum(-delta * CARBON_DENSITY[v] * cell_area_2d) * TC_TO_MTCO2

# Grouped view
flux_groups = {
    'Forest (primf+secdf)': ['primf', 'secdf'],
    'Non-forest (primn+secdn)': ['primn', 'secdn'],
    'Cropland': ['c3ann', 'c3nfx', 'c3per', 'c4ann', 'c4per'],
    'Pasture+Range': ['pastr', 'range'],
    'Urban': ['urban'],
    'Plantations': ['pltns'],
}

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Top: stacked contributions
ax = axes[0]
group_colors = ['#2e7d32', '#8d6e63', '#ff9800', '#e91e63', '#757575', '#8bc34a']
n_plot = 150  # show first 150 years of extension
bottom_pos = np.zeros(n_plot)
bottom_neg = np.zeros(n_plot)
for (label, varlist), color in zip(flux_groups.items(), group_colors):
    grp_flux = sum(flux_by_var[v][:n_plot] for v in varlist)
    pos = np.where(grp_flux > 0, grp_flux, 0)
    neg = np.where(grp_flux < 0, grp_flux, 0)
    ax.bar(full_ext_years[:n_plot], pos, bottom=bottom_pos, color=color, alpha=0.7, width=1, label=label)
    ax.bar(full_ext_years[:n_plot], neg, bottom=bottom_neg, color=color, alpha=0.7, width=1)
    bottom_pos += pos
    bottom_neg += neg

ax.axhline(0, color='k', lw=0.5)
ax.axvline(cfg.YR_AFOLU_ZERO, color='r', ls=':', lw=0.8)
ax.set_ylabel('Mt CO2/yr'); ax.set_title('AFOLU flux decomposition by land type')
ax.legend(loc='upper right', fontsize=8)
ax.set_xlim(2100, 2200)

# Bottom: net flux vs the ramp multiplier
ax = axes[1]
from src.data import afolu_ramp
ramp = afolu_ramp(full_ext_years)
ax.plot(full_ext_years[:n_plot], ext_flux[:n_plot], 'b-', lw=1.5, label='Net AFOLU flux')
ax2 = ax.twinx()
ax2.plot(full_ext_years[:n_plot], ramp[:n_plot], 'gray', ls='--', lw=1, alpha=0.6, label='Rate ramp')
ax2.set_ylabel('Rate multiplier', color='gray')
ax.axhline(0, color='k', lw=0.5)
ax.axvline(cfg.YR_AFOLU_ZERO, color='r', ls=':', lw=0.8)
ax.set_xlabel('Year'); ax.set_ylabel('Mt CO2/yr')
ax.set_title('Net flux vs rate ramp')
ax.legend(loc='upper right')

plt.tight_layout(); plt.show()

## 8. Cumulative Emissions Check

Compare cumulative AFOLU emissions from 2100 onward between the bookkeeping estimate and the IAM target. This tests whether the *integral* of the flux is consistent, not just the year-by-year values.

In [ ]:
# Cumulative from 2101 onward
cum_est = np.cumsum(ext_flux)

# IAM cumulative
afolu_ext_yrs = np.arange(2101, 2251)
afolu_ext_vals = np.array([afolu_target.loc[y] for y in afolu_ext_yrs])
cum_iam = np.cumsum(afolu_ext_vals)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(full_ext_years[:150], cum_est[:150], 'b-', lw=1.5, label='Bookkeeping estimate')
ax.plot(afolu_ext_yrs, cum_iam, 'r--', lw=1.5, label='IAM target')
ax.axvline(cfg.YR_AFOLU_ZERO, color='r', ls=':', lw=0.8)
ax.set_xlabel('Year'); ax.set_ylabel('Cumulative Mt CO2')
ax.set_title('Cumulative AFOLU emissions from 2101')
ax.legend()

# Normalised cumulative
ax = axes[1]
idx_2149_est = np.searchsorted(full_ext_years, 2149)
idx_2149_iam = np.searchsorted(afolu_ext_yrs, 2149)
cum_est_norm = cum_est[:150] / cum_est[idx_2149_est] if cum_est[idx_2149_est] != 0 else cum_est[:150]
cum_iam_norm = cum_iam / cum_iam[idx_2149_iam] if cum_iam[idx_2149_iam] != 0 else cum_iam
ax.plot(full_ext_years[:150], cum_est_norm, 'b-', lw=1.5, label='Estimate (normalised)')
ax.plot(afolu_ext_yrs, cum_iam_norm, 'r--', lw=1.5, label='IAM (normalised)')
ax.axvline(cfg.YR_AFOLU_ZERO, color='r', ls=':', lw=0.8)
ax.set_xlabel('Year'); ax.set_ylabel('Normalised cumulative')
ax.set_title('Normalised cumulative shape')
ax.legend()

plt.tight_layout(); plt.show()

# Print summary
print(f"Cumulative estimate by 2149: {cum_est[idx_2149_est]:.0f} Mt CO2")
print(f"Cumulative IAM target by 2149: {cum_iam[idx_2149_iam]:.0f} Mt CO2")

## 9. Spatial Pattern of AFOLU Flux at Key Years

Where are the largest emissions and sinks located? Map the per-cell AFOLU flux at 2110 (mid-ramp), 2130, and 2149 (end of ramp). Post-2150 flux is zero everywhere.

In [ ]:
map_years = [2110, 2130, 2149]
fig, axes = plt.subplots(1, len(map_years), figsize=(6 * len(map_years), 4))

for ax, yr in zip(axes, map_years):
    ti = np.searchsorted(ramp_years, yr)
    # Per-cell flux for this year
    flux_map = np.zeros((len(lat), len(lon)))
    for v in cfg.STATE_VARS:
        if ti == 0:
            prev = vals_2100[v]
        else:
            prev = ds_ext[v].isel(time=ti-1).values
        curr = ds_ext[v].isel(time=ti).values
        delta = curr - prev
        flux_map += -delta * CARBON_DENSITY[v] * cell_area_2d * TC_TO_MTCO2

    vmax = np.nanpercentile(np.abs(flux_map[np.isfinite(flux_map)]), 98)
    im = ax.pcolormesh(lon, lat, flux_map, cmap='RdBu_r', shading='auto',
                       vmin=-vmax, vmax=vmax)
    ax.set_title(f'AFOLU flux @ {yr} (Mt CO2/yr per cell)')
    plt.colorbar(im, ax=ax, shrink=0.7, label='Mt CO2/yr')

plt.tight_layout(); plt.show()

## 10. Sensitivity to Carbon Density Assumptions

Test how the estimated flux changes under different carbon density assumptions. We run +50% and -50% scenarios for forest carbon densities to bracket the uncertainty.

In [ ]:
# Sensitivity: scale forest carbon densities by +/-50%
forest_vars = ['primf', 'secdf', 'pltns']
scales = {'Low C (-50%)': 0.5, 'Central': 1.0, 'High C (+50%)': 1.5}

fig, ax = plt.subplots(figsize=(12, 5))

for label, scale in scales.items():
    cd_test = CARBON_DENSITY.copy()
    for fv in forest_vars:
        cd_test[fv] = int(CARBON_DENSITY[fv] * scale)

    flux_test = np.zeros(len(full_ext_years))
    for ti in range(n_ramp):
        af = 0.0
        for v in cfg.STATE_VARS:
            if ti == 0:
                prev = vals_2100[v]
            else:
                prev = ds_ext[v].isel(time=ti-1).values
            curr = ds_ext[v].isel(time=ti).values
            delta = curr - prev
            af += np.nansum(-delta * cd_test[v] * cell_area_2d) * TC_TO_MTCO2
        flux_test[ti] = af

    ax.plot(full_ext_years[:100], flux_test[:100], lw=1.5, label=label)

# IAM target overlay
afolu_sub = afolu_target.loc[2101:2200]
ax.plot(afolu_sub.index, afolu_sub.values, 'k--', lw=1.5, label='IAM target')
ax.axhline(0, color='k', lw=0.5)
ax.axvline(cfg.YR_AFOLU_ZERO, color='r', ls=':', lw=0.8)
ax.set_xlabel('Year'); ax.set_ylabel('Mt CO2/yr')
ax.set_title('Sensitivity: AFOLU flux under different forest C densities')
ax.legend(); plt.tight_layout(); plt.show()

## 11. Summary

Key findings from the AFOLU flux check:

1. **Temporal shape**: The bookkeeping estimate should decline toward zero in step with the linear ramp, matching the IAM AFOLU trajectory by construction.
2. **Magnitude**: The absolute magnitude depends on assumed carbon densities. These are rough global means and not spatially resolved, so the estimate is indicative rather than precise.
3. **Zero crossing**: Fluxes should reach (near-)zero by 2149 when the ramp hits zero and all land-use change ceases.
4. **Post-ramp**: After 2149, fluxes should be zero (no further land-use change), consistent with the frozen steady state.
5. **Sensitivity**: Forest carbon density dominates the uncertainty. Even with +/-50% variation, the temporal shape is preserved.

In [ ]:
# Clean up
ds_ext.close()
ds_input.close()
print("All datasets closed.")